**Install Tools**

In [1]:
%%bash
apt-get update -qq
apt-get install -y -qq fastqc samtools unzip wget default-jre
pip install -q multiqc ffq

Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected package at-spi2-core.
Preparing to unpack .../04-at-spi2-core_2.44.0-3_amd64.deb ...
Unpacking at-spi2-core (2.44.0-3) ...
Selecting previously unselected package openjdk-11-jre-headless:a

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


**Download Tools (HISAT2 + Subread)**

In [2]:
%%bash
# HISAT2 (FIXED)
wget -q https://cloud.biohpc.swmed.edu/index.php/s/oTtGWbWjaxsQ2Ho/download -O hisat2.zip
unzip -o -q hisat2.zip   # ← IMPORTANT: -o avoids prompt

# Create symlink safely
HISAT2_PATH=$(find /content -name hisat2 | head -n 1)
ln -sf $HISAT2_PATH /usr/local/bin/hisat2

# Subread (FIXED)
wget -q https://sourceforge.net/projects/subread/files/subread-2.0.6/subread-2.0.6-Linux-x86_64.tar.gz/download -O subread.tar.gz
tar -xzf subread.tar.gz

# Create symlink safely
FEATURE_PATH=$(find /content -name featureCounts | head -n 1)
ln -sf $FEATURE_PATH /usr/local/bin/featureCounts

**Download FASTQ from SRA**

In [3]:
import subprocess, json, urllib.request, os

SRR = "SRR1039508"  # change if needed
os.makedirs("fastq", exist_ok=True)

# Get FASTQ links
result = subprocess.run(["ffq", "--ftp", SRR], capture_output=True, text=True)
links = json.loads(result.stdout)

# Download files
for entry in links:
    url = entry["url"]
    fname = url.split("/")[-1]
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(url, f"fastq/{fname}")
    print(f"Done: {fname}")

Done: SRR1039508_1.fastq.gz
Done: SRR1039508_2.fastq.gz


**Quality Control (Raw Reads)**

In [4]:
%%bash
mkdir -p qc_raw
fastqc fastq/*.fastq* -o qc_raw -t 2
multiqc qc_raw -o qc_raw

Analysis complete for SRR1039508_2.fastq.gz
Analysis complete for SRR1039508_1.fastq.gz


Started analysis of SRR1039508_1.fastq.gz
Started analysis of SRR1039508_2.fastq.gz
Approx 5% complete for SRR1039508_1.fastq.gz
Approx 5% complete for SRR1039508_2.fastq.gz
Approx 10% complete for SRR1039508_1.fastq.gz
Approx 10% complete for SRR1039508_2.fastq.gz
Approx 15% complete for SRR1039508_1.fastq.gz
Approx 15% complete for SRR1039508_2.fastq.gz
Approx 20% complete for SRR1039508_1.fastq.gz
Approx 20% complete for SRR1039508_2.fastq.gz
Approx 25% complete for SRR1039508_1.fastq.gz
Approx 25% complete for SRR1039508_2.fastq.gz
Approx 30% complete for SRR1039508_1.fastq.gz
Approx 30% complete for SRR1039508_2.fastq.gz
Approx 35% complete for SRR1039508_1.fastq.gz
Approx 35% complete for SRR1039508_2.fastq.gz
Approx 40% complete for SRR1039508_1.fastq.gz
Approx 40% complete for SRR1039508_2.fastq.gz
Approx 45% complete for SRR1039508_1.fastq.gz
Approx 45% complete for SRR1039508_2.fastq.gz
Approx 50% complete for SRR1039508_1.fastq.gz
Approx 50% complete for SRR1039508_2.fastq.g

**Download Adapter File**

In [5]:
%%bash
wget -q https://raw.githubusercontent.com/usadellab/Trimmomatic/main/adapters/TruSeq3-PE.fa

**Trimming Reads**

In [6]:
!ls -R | grep -i trimmomatic

In [7]:
%%bash
# Clean old install (important in Colab)
rm -rf trimmomatic-0.39*

# Download again
wget -q https://github.com/usadellab/Trimmomatic/releases/download/v0.39/trimmomatic-0.39.zip
unzip -o -q trimmomatic-0.39.zip

# Download adapters
wget -q https://raw.githubusercontent.com/usadellab/Trimmomatic/main/adapters/TruSeq3-PE.fa

In [8]:
%%bash
mkdir -p trimmed

# Find Trimmomatic jar automatically
TRIM_JAR=$(find /content -name "*trimmomatic*.jar" | head -n 1)

echo "Using:" $TRIM_JAR

R1=$(ls fastq/*_1.fastq* | head -n 1)
R2=$(ls fastq/*_2.fastq* | head -n 1)

java -jar $TRIM_JAR PE \
  $R1 $R2 \
  trimmed/R1_paired.fastq trimmed/R1_unpaired.fastq \
  trimmed/R2_paired.fastq trimmed/R2_unpaired.fastq \
  ILLUMINACLIP:TruSeq3-PE.fa:2:30:10 \
  LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

Using: /content/Trimmomatic-0.39/trimmomatic-0.39.jar


TrimmomaticPE: Started with arguments:
 fastq/SRR1039508_1.fastq.gz fastq/SRR1039508_2.fastq.gz trimmed/R1_paired.fastq trimmed/R1_unpaired.fastq trimmed/R2_paired.fastq trimmed/R2_unpaired.fastq ILLUMINACLIP:TruSeq3-PE.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Multiple cores found: Using 2 threads
Using PrefixPair: 'TACACTCTTTCCCTACACGACGCTCTTCCGATCT' and 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Quality encoding detected as phred33
Input Read Pairs: 22935521 Both Surviving: 22141244 (96.54%) Forward Only Surviving: 351667 (1.53%) Reverse Only Surviving: 340132 (1.48%) Dropped: 102478 (0.45%)
TrimmomaticPE: Completed successfully


**Download HISAT2 Genome Index**

In [9]:
%%bash
wget -q https://genome-idx.s3.amazonaws.com/hisat/grch38_genome.tar.gz
tar -xzf grch38_genome.tar.gz

**Alignment (HISAT2)**

In [10]:
%%bash
hisat2 -x grch38/genome \
  -1 trimmed/R1_paired.fastq \
  -2 trimmed/R2_paired.fastq \
  -S aligned.sam \
  --summary-file alignment_summary.txt

22141244 reads; of these:
  22141244 (100.00%) were paired; of these:
    750201 (3.39%) aligned concordantly 0 times
    20558383 (92.85%) aligned concordantly exactly 1 time
    832660 (3.76%) aligned concordantly >1 times
    ----
    750201 pairs aligned concordantly 0 times; of these:
      118038 (15.73%) aligned discordantly 1 time
    ----
    632163 pairs aligned 0 times concordantly or discordantly; of these:
      1264326 mates make up the pairs; of these:
        749138 (59.25%) aligned 0 times
        454406 (35.94%) aligned exactly 1 time
        60782 (4.81%) aligned >1 times
98.31% overall alignment rate


**Convert + Sort + Index BAM**

In [11]:
%%bash
samtools view -@ 2 -bS aligned.sam > aligned.bam
samtools sort -@ 2 aligned.bam -o aligned_sorted.bam
samtools index aligned_sorted.bam

[bam_sort_core] merging from 14 files and 2 in-memory blocks...


**Download Annotation (GTF)**

In [12]:
%%bash
wget -q ftp://ftp.ensembl.org/pub/release-110/gtf/homo_sapiens/Homo_sapiens.GRCh38.110.gtf.gz
gunzip Homo_sapiens.GRCh38.110.gtf.gz

**Gene Counting (featureCounts)**

In [13]:
%%bash
featureCounts -T 2 -p -B -C \
  -s 0 \
  -a Homo_sapiens.GRCh38.110.gtf \
  -o counts.txt aligned_sorted.bam


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           aligned_sorted.bam                               ||
||                                                                            ||
||             Output file : counts.txt                          

**View Final Results**

In [14]:
import pandas as pd

df = pd.read_csv("counts.txt", sep="\t", comment="#")
df.head()

,Geneid,Chr,Start,End,Strand,Length,aligned_sorted.bam
0,ENSG00000279928,1;1;1;1;1,182696;183132;183494;183740;183981,182746;183216;183571;183901;184174,+;+;+;+;+,570,1
1,ENSG00000228037,1;1;1,2581560;2583370;2584125,2581650;2583495;2584533,+;+;+,626,2
2,ENSG00000142611,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;...,3069168;3069183;3069197;3069203;3069211;318612...,3069296;3069296;3069296;3069296;3069296;318647...,+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;...,9767,155
3,ENSG00000284616,1;1;1;1,5301928;5303328;5304401;5306942,5302004;5303393;5305029;5307394,-;-;-;-,1225,0
4,ENSG00000157911,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;...,2403964;2403974;2404061;2404613;2404838;240503...,2405834;2405834;2404371;2404891;2405828;240583...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,3782,400


In [15]:
# Example: adjust column names based on your file
counts = df.iloc[:, -1:]   # last column = aligned_sorted.bam

# Add gene IDs
counts.index = df.iloc[:, 0]

counts.head()

,aligned_sorted.bam
Geneid,
ENSG00000279928,1
ENSG00000228037,2
ENSG00000142611,155
ENSG00000284616,0
ENSG00000157911,400
